[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/shells/shells.ipynb)

# Discrete shells

A thin elastic sheet resists two things: stretching, which changes lengths along it, and bending, which changes how it curves. Modelled as a triangle mesh, after Grinspun, Hirani, Desbrun and Schröder, "Discrete shells" (2003), it stretches along its edges and bends at its hinges, the edges with a face on either side. A sheet whose rest state does not fit in space must buckle: a disc that grows towards its rim becomes a saddle, and a strip with the crossed natural curvature of a seed pod's two layers twists into a ribbon.

In the projective geometric algebra of space, an edge is the line joining its two ends, a face's plane is the join of a hinge's line with the corner across it, and the fold of one face onto the other is a motor about the hinge.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from examples.animation import save_figure, show_animation
from examples.mechanics.shells import render
from examples.mechanics.shells.core import Mesh, Sheet, as_scalar, flat_mesh, ga, hinges, lattice, mv

np.set_printoptions(precision=3, suppress=True)

Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
Direction = Point.derive.degenerate
# Lines joining points: edges, hinges and the lines forces act along.
Line = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
# The form of a natural curvature, on unit lines in the sheet: the curvature across them.
Curvature = ga.gatype((Scalar, Line, Line))                  # Scalar <- (Line, Line)

## 1. Lengths and bending angles

An edge is the line joining its two ends, and a line joining unit points carries their distance as its norm. A hinge's two faces lie in the planes joining its line with the corner across it on either side. The product of the two unit planes is twice the turn about their common line, so its square root is the motor that folds one face onto the other, and the logarithm of that motor is half the bending angle times the hinge's unit line.

The sheet lives on a mesh, and every quantity is a field over the mesh's vertices, edges, faces or hinges. The mesh's reading maps carry a vertex field to the ends of every edge or the corners of every face; `hinges` finds each edge between two faces with the corners across it, from the mesh's alignment of every face's edges with the corners they face.

In [ ]:
def hinge_motors(ends: Point, tips: Point, across: Point, opposite: Point) -> tuple[Line, Plane, Plane, Motor]:
    # Every hinge's line, the planes of its two faces, and the motor folding the first face onto the second.
    line = ends & tips                                                           # Line[H]
    # Both planes wound as the first face, so that a flat hinge folds by nothing.
    first, second = line & across, -(line & opposite)                            # Plane[H] each
    # The product of two unit planes is twice the turn about their common line: its square root folds one onto the other.
    return line, first, second, (second.normalized() * first.normalized()).square_root()


def bending_angles(line: Line, motor: Motor) -> Scalar:
    # Twice the logarithm of every hinge's motor, measured along its unit line.
    return -2 * (motor.log() | line.normalized())

## 2. Energy and forces

The sheet stores energy as the squared stretch of every edge, against its rest length, and the squared bend of every hinge, against its rest angle. The forces are the energy's exact downhill direction. Every edge pulls its ends together or apart along its line. A hinge bent away from its rest angle twists its two faces about its line, equally and oppositely: each face pushes the corner across the hinge square to the face, by the twist over the corner's distance from the hinge, `((plane | c) ^ w)` being the direction of the line through the corner square to the plane, and the hinge's two ends take the opposite push, shared by the lever rule, by where the corner lies along the hinge.

As internal forces they balance. The line along which a force acts is the join of its vertex with the force, and these lines add up to a forque, a force and a moment at once; for the sheet's own forces it is zero.

In the notation of the discrete shells paper the bending energy is $\sum_e (\theta_e - \bar\theta_e)^2 \, 3 \bar\ell_e^2 / \bar A_e$ with $\bar A_e$ the summed area of the two faces, and the gradient of the angle at the corner across the hinge is $n / h$, the unit normal over the corner's height above the hinge.

In [ ]:
def energy(sheet: Sheet, points: Point) -> Scalar:
    tails, heads = (end * points for end in sheet.mesh.edge_ends)
    stretched = (tails & heads).norm() - sheet.lengths                           # Scalar[E]
    line, _, _, motor = hinge_motors(*(corner * points for corner in sheet.corners))
    bent = bending_angles(line, motor) - sheet.angles                             # Scalar[H]
    return ((sheet.stretching * stretched * stretched / sheet.lengths).sites.sum() + (sheet.bending * bent * bent).sites.sum()) / 2


def forces(sheet: Sheet, points: Point) -> Direction:
    # The edges' springs pulling their ends together or apart.
    tails, heads = (end * points for end in sheet.mesh.edge_ends)
    lengths = (tails & heads).norm()                                             # Scalar[E]
    pull = (heads - tails) * (sheet.stretching * (lengths - sheet.lengths) / (sheet.lengths * lengths))   # Direction[E]

    # The hinges twisting their faces back towards their rest angles.
    corners = [corner * points for corner in sheet.corners]                     # Point[H] each
    ends, _, across, opposite = corners
    line, first, second, motor = hinge_motors(*corners)
    moment = line.norm() * sheet.bending * (bending_angles(line, motor) - sheet.angles)   # Scalar[H]
    # Each face pushes the corner across the hinge square to its plane, along `(plane | corner) ^ w`, by the
    # moment over the corner's distance from the hinge.
    pushes = [((plane | corner) ^ mv.w) * (moment / plane.norm_squared()) for plane, corner in ((first, across), (second, opposite))]
    # The hinge's ends take the opposite pushes, shared by the lever rule: by where each corner lies along the hinge.
    shares = [((ends & corner) | line) / (line | line) for corner in (across, opposite)]
    at_tips = -(pushes[0] * shares[0] + pushes[1] * shares[1])                   # Direction[H]
    at_ends = -(pushes[0] + pushes[1]) - at_tips                                 # Direction[H]

    # The incidence's reverse sums every edge's pull at its two ends, and each reading map's reverse sums
    # the hinges' pushes at their corners.
    at_ends_map, at_tips_map, across_map, opposite_map = sheet.corners
    return (-(~sheet.mesh.d0 * pull) + ~at_ends_map * at_ends + ~at_tips_map * at_tips
            + ~across_map * pushes[0] + ~opposite_map * pushes[1])


def forque(points: Point, pushes: Direction) -> Line:
    # The sum of the lines along which the forces act, each through its vertex.
    return (points & pushes).sites.sum()

## 3. The rest state, and relaxing towards it

The rest angles come from a natural curvature, a form on lines: bound with a hinge's unit line on both sides it gives the curvature across the hinge, and times the distance between the two faces' centres, a third of the sum of the corners' distances from the hinge, the angle the hinge would bend by on a surface of that curvature. The bending angles count towards the sheet's top side, read off the planes through its faces. The sheet then moves under its own forces from nearly flat, losing a share of its velocity every step, until it settles.

In [ ]:
def sheet(mesh: Mesh, lengths: Scalar, curvature: Curvature, top: Plane, thickness: float) -> Sheet:
    # A sheet of the given thickness, of unit stiffness and density, at rest at the mesh's vertices.
    points = mesh.vertices
    corners = tuple(mesh.reading(index) for index in hinges(mesh).T)             # [H, V] each
    line, first, second, _ = hinge_motors(*(corner * points for corner in corners))
    # The curvature across each hinge, times the distance between its faces' centres, a third of the sum of
    # the corners' distances from the hinge, is the angle the hinge bends by on a surface of that curvature.
    unit = line.normalized()
    angles = curvature(unit, unit) * (first.norm() + second.norm()) / (3 * line.norm())   # Scalar[H]
    # The plane through each face's corners, wound as the face, has twice the face's area as its norm; read
    # against the top side, it says which way the bending angles count.
    corner_points = [corner * points for corner in mesh.face_corners]           # Point[F] each
    planes = corner_points[0] & corner_points[1] & corner_points[2]              # Plane[F]
    facing = (planes | top).sites.sum()                                          # [] Scalar
    # Each vertex carries a third of the mass of every face at it.
    doubled = planes.norm()                                                      # Scalar[F]
    masses = ~mesh.at_corners(stack([doubled] * 3)) * as_scalar(np.ones(len(mesh.faces))).field() * (thickness / 6)
    return Sheet(mesh=mesh, corners=corners, lengths=lengths, angles=angles * (facing / facing.norm()),
                 stretching=thickness, bending=thickness ** 3 / 2 * -(line | line) / (first.norm() + second.norm()),
                 masses=masses)


def relax(sheet: Sheet, points: Point, step: float, damping: float, steps: int, every: int):
    # The points every `every` steps, and at the end, moving from rest under the sheet's own forces.
    velocity = points - points                                                   # Direction[V]
    for count in range(steps + 1):
        if count % every == 0 or count == steps:
            yield points
        velocity = (velocity + forces(sheet, points) * (step / sheet.masses)) * (1 - damping)
        points = points + velocity * step


def roughened(flat: np.ndarray, roughness: float, seed: int) -> Point:
    # A flat sheet's points, lifted by small random heights to let it leave its plane.
    heights = roughness * np.random.default_rng(seed).normal(size=(len(flat), 1))
    return mv.point(np.concatenate([flat, heights], axis=-1)).field()


damping, steps, every = 0.01, 2000, 25                # the share of velocity lost every step; the steps; per frame
roughness, seed = 1e-3, 0

## 4. A disc that grows towards its rim

Every edge's rest length grows with the square of its middle's distance from the centre, by 30 percent at the rim. A flat disc has too little rim for those lengths, so the disc leaves its plane, and with no natural curvature it settles into a saddle.

In [ ]:
disc_radius, disc_spacing, growth, disc_thickness, disc_step = 1.0, 0.1, 0.3, 0.03, 0.02

flat = lattice(disc_spacing, disc_radius)
flat = flat[np.linalg.norm(flat, axis=-1) <= disc_radius + 1e-9]
disc_mesh = flat_mesh(flat, disc_spacing)
tails, heads = (end * disc_mesh.vertices for end in disc_mesh.edge_ends)       # Point[E] each
# Each edge grows with the square of its middle's distance from the centre, in disc radii.
distance = (mv.w.dual() & ((tails + heads) / 2)).norm() / disc_radius          # Scalar[E]
disc_lengths = (tails & heads).norm() * (1 + growth * distance * distance)     # Scalar[E]
disc = sheet(disc_mesh, disc_lengths, 0 * (Line | Line), mv.z, disc_thickness)
disc_frames = list(relax(disc, roughened(flat, roughness, seed), disc_step, damping, steps, every))
show_animation(render.animate(disc_mesh.faces, disc_frames), "shells_disc", render.DURATION_MS)

## 5. A seed pod's strip

A seed pod's wall is two layers that each want to curve along their own direction, the two crossing the strip at 45 degrees either way. The natural curvature curves positively across one direction and negatively across the other, a saddle that no unstretched surface can take. Neither layer wins along the whole strip, and it twists into a ribbon.

In [ ]:
strip_length, strip_width, strip_spacing, strip_thickness, strip_step = 2.0, 0.3, 0.08, 0.05, 0.012
natural_curvature, pod_angle = 6.0, np.pi / 4

flat = lattice(strip_spacing, strip_length)
flat = flat[(np.abs(flat[:, 0]) <= strip_length / 2) & (np.abs(flat[:, 1]) <= strip_width / 2)]
strip_mesh = flat_mesh(flat, strip_spacing)
tails, heads = (end * strip_mesh.vertices for end in strip_mesh.edge_ends)     # Point[E] each
# The two layers curve along lines through the centre, turned either way from the strip's axis.
axis = mv.w.dual() & mv.x.dual()                                                   # [] Line
along, across = (((mv.xy * (-sense * pod_angle / 2)).exp() >> axis) for sense in (1, -1))
curvature = natural_curvature * ((across | Line) * (across | Line) - (along | Line) * (along | Line))   # [] Curvature
strip = sheet(strip_mesh, (tails & heads).norm(), curvature, mv.z, strip_thickness)
strip_frames = list(relax(strip, roughened(flat, roughness, seed), strip_step, damping, steps, every))
show_animation(render.animate(strip_mesh.faces, strip_frames), "shells_strip", render.DURATION_MS)

The settled disc and strip side by side, shaded by the directions their faces face.

In [ ]:
save_figure(render.draw_settled([(disc_mesh.faces, disc_frames[-1]), (strip_mesh.faces, strip_frames[-1])]), "shells_settled");

In [ ]:
# checks
for shell, frames, rise in ((disc, disc_frames, 0.3), (strip, strip_frames, 0.1)):
    settled = frames[-1]
    # The forces inside the sheet balance: their lines add up to no force and no moment.
    total = forque(settled, forces(shell, settled))
    assert (-(total | total) - (total.dual() | total.dual())).to_array() < 1e-30
    # The sheet has left its plane.
    assert np.ptp((settled & mv.z).batch().to_array()) > rise
# The disc has settled close to its grown lengths.
tails, heads = (end * disc_frames[-1] for end in disc_mesh.edge_ends)
strain = ((tails & heads).norm() / disc_lengths - 1).batch().to_array()
assert np.median(np.abs(strain)) < 0.02